In [5]:
from ad_afqmc_prototype import config
from pyscf import gto, scf, cc
config.configure_once()
from ad_afqmc_prototype.afqmc import Afqmc

a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'

atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,basis=basis,spin=spin*nc,unit=unit,verbose=4)
mol.build()

mf = scf.RHF(mol)
mf.kernel()

mycc = cc.CCSD(mf)
mycc.frozen = 0  # freeze O 1s core
mycc.kernel()
et = mycc.ccsd_t()  # for comparison
print(f"CCSD(T) total energy: {mycc.e_tot + et}")

afqmc = Afqmc(mycc)
afqmc.n_walkers = 100  # number of walkers
afqmc.n_eql_blocks = 10  # number of equilibration blocks
afqmc.n_blocks = 100  # number of sampling blocks
afqmc.mixed_precision = False
mean, err = afqmc.kernel()

System: uname_result(system='Linux', node='sharmagroup-rn', release='6.17.0-19-generic', version='#19~24.04.2-Ubuntu SMP PREEMPT_DYNAMIC Fri Mar  6 23:08:46 UTC 2', machine='x86_64')  Threads 16
Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
numpy 2.3.1  scipy 1.16.2  h5py 3.14.0
Date: Fri Mar 20 17:59:59 2026
PySCF version 2.12.1
PySCF path  /home/sharmagroup/sharmagroup/pyscf
GIT ORIG_HEAD 3d1768f5e33b144b606c3d2c81c12ee54d794501
GIT HEAD (branch master) f0861da51f017364d8bbaa20b742a94f3733305f

[ENV] PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = b
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 H      0.000000000000   0.000000000000   0.000000000000 AA    

In [9]:
from pyscf import gto, scf, cc
from ad_afqmc_prototype import config
# config.configure_once()
from ad_afqmc_prototype.afqmc import Afqmc

a = 2 # bond length in a cluster
d = 100 # distance between each cluster
unit = 'b' # unit of length
na = 2 # size of a cluster (monomer)
nc = 1 # set as integer multiple of monomers
spin = 0 # spin per monomer
frozen = 0 # frozen orbital per monomer
elmt = 'H'
basis = 'sto6g'

atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,basis=basis,spin=spin*nc,unit=unit,verbose=4)
mol.build()


mf = scf.RHF(mol)
mf.kernel()

mycc = cc.CCSD(mf)
mycc.kernel()

afqmc = Afqmc(mycc)
afqmc.n_walkers = 100  # number of walkers
afqmc.n_eql_blocks = 10  # number of equilibration blocks
afqmc.n_blocks = 100  # number of sampling blocks
mean, err = afqmc.kernel()

ImportError: cannot import name 'configure_once' from 'ad_afqmc_prototype.config' (/home/sharmagroup/sharmagroup/build_afqmc_prototype/ad_afqmc_prototype/ad_afqmc_prototype/config.py)

In [4]:
import numpy as np

from ad_afqmc_prototype import driver, config
from ad_afqmc_prototype.prep import integrals
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.ham.chol import HamChol
from ad_afqmc_prototype.prop.afqmc import make_prop_ops
from ad_afqmc_prototype.prop.blocks import block
from ad_afqmc_prototype.prop.types import QmcParams
from ad_afqmc_prototype.prep.pyscf_interface import get_integrals, get_cisd
from ad_afqmc_prototype.trial.cisd import CisdTrial, make_cisd_trial_ops
from ad_afqmc_prototype.meas.cisd import make_cisd_meas_ops

# class Cisd:
#     def __init__(self, cc):
config.setup_jax()

mol = mycc.mol
h0, h1, chol = get_integrals(mycc._scf)
ci1, ci2 = get_cisd(mycc)

sys = System(norb=mol.nao, nelec=mol.nelec, walker_kind="restricted")
ham_data = HamChol(h0, h1, chol)
trial_data = CisdTrial(ci1, ci2)
trial_ops = make_cisd_trial_ops(sys=sys)
meas_ops = make_cisd_meas_ops(sys=sys)
prop_ops = make_prop_ops(ham_data.basis, sys.walker_kind)
params = QmcParams(
    n_eql_blocks=20, n_blocks=200, seed=np.random.randint(0, int(1e6))
)
block_fn = block
# sys = sys
# ham_data = ham_data

driver.run_qmc_energy(
            sys=sys,
            params=params,
            ham_data=ham_data,
            trial_ops=trial_ops,
            trial_data=trial_data,
            meas_ops=meas_ops,
            prop_ops=prop_ops,
            block_fn=block_fn,
        )

Starting QMC driver...
Parameters:
QmcParams(dt=0.005,
          n_chunks=1,
          n_exp_terms=6,
          pop_control_damping=0.1,
          weight_floor=0.001,
          weight_cap=100.0,
          n_prop_steps=50,
          shift_ema=0.1,
          n_eql_blocks=20,
          n_blocks=200,
          n_walkers=200,
          seed=451688)


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/20]   -1.0960712820  2.000000e+02           0       0.0
[eql    4/20]   -1.0960712830  2.000279e+02           0       0.5
[eql    8/20]   -1.0960712850  1.999877e+02           0       1.0
[eql   12/20]   -1.0960712829  1.999939e+02           0       1.0
[eql   16/20]   -1.0960712835  2.000080e+02           0       1.0
[eql   20/20]   -1.0960712829  2.000100e+02           0       1.1

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk   20/200]   -1.0960712831   9.535e-10   -1.096071

(-1.0960712837814546,
 7.81050712785987e-10,
 Array([-1.09607128, -1.09607128, -1.09607128, -1.09607128, -1.09607128,
        -1.09607128, -1.09607128, -1.09607129, -1.09607129, -1.09607128,
        -1.09607128, -1.09607128, -1.09607128, -1.09607128, -1.09607129,
        -1.09607128, -1.09607128, -1.09607128, -1.09607128, -1.09607128,
        -1.09607128, -1.09607128, -1.09607128, -1.09607128, -1.09607128,
        -1.09607128, -1.09607128, -1.09607129, -1.09607128, -1.09607129,
        -1.09607127, -1.09607129, -1.09607128, -1.09607129, -1.09607128,
        -1.09607128, -1.09607128, -1.09607128, -1.09607129, -1.09607128,
        -1.09607128, -1.09607129, -1.09607129, -1.09607128, -1.09607129,
        -1.09607128, -1.09607128, -1.09607129, -1.09607128, -1.09607128,
        -1.09607129, -1.09607128, -1.09607128, -1.09607128, -1.09607128,
        -1.09607128, -1.09607128, -1.09607128, -1.09607128, -1.09607128,
        -1.09607128, -1.09607128, -1.09607129, -1.09607128, -1.09607128,
      

In [5]:
import time
from functools import partial
from pprint import pprint
from typing import Any, Callable

import jax
import jax.numpy as jnp
from jax import lax
from jax.sharding import Mesh, NamedSharding
from jax.sharding import PartitionSpec as P

from ad_afqmc_prototype.core.ops import MeasOps, TrialOps
from ad_afqmc_prototype.core.system import System
from ad_afqmc_prototype.prop.blocks import BlockFn
from ad_afqmc_prototype.prop.types import PropOps, PropState, QmcParams
from ad_afqmc_prototype.stat_utils import blocking_analysis_ratio, reject_outliers
from ad_afqmc_prototype.walkers import stochastic_reconfiguration

# def run_qmc_energy(
#     *,
#     sys: System,
#     params: QmcParams,
#     ham_data: Any,
#     trial_data: Any,
#     meas_ops: MeasOps,
#     trial_ops: TrialOps,
#     prop_ops: PropOps,
#     block_fn: BlockFn,
#     state: PropState | None = None,
#     meas_ctx: Any | None = None,
#     target_error: float | None = None,
#     mesh: Mesh | None = None,
# ) -> tuple[jax.Array, jax.Array, jax.Array, jax.Array]:
#     """
#     equilibration blocks then sampling blocks.

#     Returns:
#       (mean_energy, stderr, block_energies, block_weights)
#     """

# driver.run_qmc_energy(
#             sys=sys,
#             params=params,
#             ham_data=ham_data,
#             trial_ops=trial_ops,
#             trial_data=trial_data,
#             meas_ops=meas_ops,
#             prop_ops=prop_ops,
#             block_fn=block_fn,
#         )

meas_ctx = None
state = None
mesh = None

print("Starting QMC driver...")
print(f"Parameters:")
pprint(params)
print("")
# build ctx
prop_ctx = prop_ops.build_prop_ctx(ham_data, trial_ops.get_rdm1(trial_data), params)
if meas_ctx is None:
    meas_ctx = meas_ops.build_meas_ctx(ham_data, trial_data)
if state is None:
    state = prop_ops.init_prop_state(
        sys=sys,
        ham_data=ham_data,
        trial_ops=trial_ops,
        trial_data=trial_data,
        meas_ops=meas_ops,
        params=params,
        mesh=mesh,
    )

if mesh is None or mesh.size == 1:
    block_fn_sr = block_fn
else:
    data_sh = NamedSharding(mesh, P("data"))
    sr_sharded = partial(stochastic_reconfiguration, data_sharding=data_sh)
    block_fn_sr = partial(block_fn, sr_fn=sr_sharded)

run_blocks = driver.make_run_blocks(
    block_fn=block_fn_sr,
    sys=sys,
    params=params,
    trial_ops=trial_ops,
    meas_ops=meas_ops,
    prop_ops=prop_ops,
)

t0 = time.perf_counter()
t_mark = t0

print_every = params.n_eql_blocks // 5 if params.n_eql_blocks >= 5 else 0
block_e_eq = []
block_w_eq = []
block_e_eq.append(state.e_estimate)
block_w_eq.append(jnp.sum(state.weights))
print("\nEquilibration:\n")
if print_every:
    print(
        f"{'':4s}"
        f"{'block':>9s}  "
        f"{'E_blk':>14s}  "
        f"{'W':>12s}   "
        f"{'nodes':>10s}  "
        f"{'t[s]':>8s}"
    )
print(
    f"[eql {0:4d}/{params.n_eql_blocks}]  "
    f"{float(state.e_estimate):14.10f}  "
    f"{float(jnp.sum(state.weights)):12.6e}  "
    f"{int(state.node_encounters):10d}  "
    f"{0.0:8.1f}"
)
chunk = print_every if print_every > 0 else 1
for start in range(0, params.n_eql_blocks, chunk):
    n = min(chunk, params.n_eql_blocks - start)
    state, e_chunk, w_chunk = run_blocks(
        state,
        ham_data=ham_data,
        trial_data=trial_data,
        meas_ctx=meas_ctx,
        prop_ctx=prop_ctx,
        n_blocks=n,
    )
    block_e_eq.extend(e_chunk.tolist())
    block_w_eq.extend(w_chunk.tolist())
    w_chunk_avg = jnp.mean(w_chunk)
    e_chunk_avg = jnp.mean(e_chunk * w_chunk) / w_chunk_avg
    elapsed = time.perf_counter() - t0
    print(
        f"[eql {start + n:4d}/{params.n_eql_blocks}]  "
        f"{float(e_chunk_avg):14.10f}  "
        f"{float(w_chunk_avg):12.6e}  "
        f"{int(state.node_encounters):10d}  "
        f"{elapsed:8.1f}"
    )
block_e_eq = jnp.asarray(block_e_eq)
block_w_eq = jnp.asarray(block_w_eq)


Starting QMC driver...
Parameters:
QmcParams(dt=0.005,
          n_chunks=1,
          n_exp_terms=6,
          pop_control_damping=0.1,
          weight_floor=0.001,
          weight_cap=100.0,
          n_prop_steps=50,
          shift_ema=0.1,
          n_eql_blocks=20,
          n_blocks=200,
          n_walkers=200,
          seed=451688)


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/20]   -1.0960712820  2.000000e+02           0       0.0
[eql    4/20]   -1.0960712830  2.000279e+02           0       0.5
[eql    8/20]   -1.0960712850  1.999877e+02           0       0.9
[eql   12/20]   -1.0960712829  1.999939e+02           0       1.0
[eql   16/20]   -1.0960712835  2.000080e+02           0       1.0
[eql   20/20]   -1.0960712829  2.000100e+02           0       1.0
